# 04 — Training & evaluation across all split schemes — CSE-CIC-IDS2018 — SVM-RFE study

Identical protocol to the base study: 5 models × schemes `70_30`, `80_20`, `90_10`,
`70_20_10` (train/val/test) and `cv5` (5-fold stratified CV), on the top 40 features
selected by **SVM-RFE**. All splits stratified with the same seed and saved to `splits/`.
Each pipeline scales and SMOTE-balances its own training portion only. A (scheme, model)
whose `metrics.json` already exists is skipped — restarts resume.

In [1]:
DATASET = "cic2018"
FT_ROOT = "ft40_rfe"
TARGET_PER_CLASS = 50_000   # per-class row cap for the in-pipeline under/over sampling
MODELS_TO_RUN = None        # None = all 5; or e.g. ["svm", "knn"]
RUN_CV = True

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 20:27:54,134 INFO nb04: 1586902 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build (or load) all split-scheme indices — stratified, fixed seed, saved to `splits/`

In [3]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)  # 2/9 of 90% = 20%
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 20:27:56,090 INFO nb04: 70_30: train=1110831 / test=476071
2026-07-06 20:27:57,021 INFO nb04: 80_20: train=1269521 / test=317381
2026-07-06 20:27:57,943 INFO nb04: 90_10: train=1428211 / test=158691
2026-07-06 20:27:58,809 INFO nb04: 70_20_10: train=1110830 / val=317381 / test=158691


## 2. Train + evaluate every (scheme × model)

In [4]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 20:27:58,820 INFO nb04: START all holdout schemes
2026-07-06 20:27:58,821 INFO nb04: START scheme 70_30
2026-07-06 20:28:14,112 INFO nb04: 70_30/decision_tree: fit on 1110831 rows in 15.3s
2026-07-06 20:28:16,160 INFO nb04: 70_30/decision_tree [test]: acc=0.7310 f1_macro=0.7032 roc_auc=0.9205 det=0.000ms


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.8s


2026-07-06 20:28:50,666 INFO nb04: 70_30/random_forest: fit on 1110831 rows in 34.5s


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.9s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.1s finished


2026-07-06 20:28:59,599 INFO nb04: 70_30/random_forest [test]: acc=0.7592 f1_macro=0.7042 roc_auc=0.9525 det=0.006ms
2026-07-06 20:29:12,128 INFO nb04: 70_30/svm: fit on 1110831 rows in 12.5s
2026-07-06 20:29:16,114 INFO nb04: 70_30/svm [test]: acc=0.6158 f1_macro=0.5468 roc_auc=0.9045 det=0.001ms
2026-07-06 20:29:21,530 INFO nb04: 70_30/naive_bayes: fit on 1110831 rows in 5.4s
2026-07-06 20:29:25,067 INFO nb04: 70_30/naive_bayes [test]: acc=0.4584 f1_macro=0.4665 roc_auc=0.8605 det=0.001ms
2026-07-06 20:29:30,256 INFO nb04: 70_30/knn: fit on 1110831 rows in 5.2s
2026-07-06 20:41:53,424 INFO nb04: 70_30/knn [test]: acc=0.7377 f1_macro=0.7011 roc_auc=0.9334 det=0.538ms
2026-07-06 20:41:53,425 INFO nb04: END   scheme 70_30 (834.6s)
2026-07-06 20:41:53,426 INFO nb04: START scheme 80_20
2026-07-06 20:42:08,368 INFO nb04: 80_20/decision_tree: fit on 1269521 rows in 14.9s
2026-07-06 20:42:09,636 INFO nb04: 80_20/decision_tree [test]: acc=0.7355 f1_macro=0.7090 roc_auc=0.9211 det=0.000ms


...

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   31.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.7s finished


2026-07-06 20:57:58,532 INFO nb04: 70_20_10/random_forest [test]: acc=0.7598 f1_macro=0.6670 roc_auc=0.9526 det=0.006ms


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.4s finished


2026-07-06 20:58:04,205 INFO nb04: 70_20_10/random_forest [validation]: acc=0.7590 f1_macro=0.7033 roc_auc=0.9522 det=0.007ms
2026-07-06 20:58:14,726 INFO nb04: 70_20_10/svm: fit on 1110830 rows in 10.5s
2026-07-06 20:58:16,122 INFO nb04: 70_20_10/svm [test]: acc=0.6132 f1_macro=0.5171 roc_auc=0.9064 det=0.001ms
2026-07-06 20:58:18,644 INFO nb04: 70_20_10/svm [validation]: acc=0.6138 f1_macro=0.5234 roc_auc=0.9077 det=0.001ms
2026-07-06 20:58:23,620 INFO nb04: 70_20_10/naive_bayes: fit on 1110830 rows in 5.0s
2026-07-06 20:58:24,737 INFO nb04: 70_20_10/naive_bayes [test]: acc=0.7542 f1_macro=0.4717 roc_auc=0.8632 det=0.001ms
2026-07-06 20:58:26,811 INFO nb04: 70_20_10/naive_bayes [validation]: acc=0.7552 f1_macro=0.4683 roc_auc=0.8629 det=0.001ms
2026-07-06 20:58:31,729 INFO nb04: 70_20_10/knn: fit on 1110830 rows in 4.9s
2026-07-06 21:02:41,503 INFO nb04: 70_20_10/knn [test]: acc=0.7413 f1_macro=0.6697 roc_auc=0.9339 det=0.525ms
2026-07-06 21:10:45,588 INFO nb04: 70_20_10/knn [validat

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,15.289,0.7310,0.6762,0.8499,0.7032,0.9205,0.0003,476071
1,70_30,random_forest,34.504,0.7592,0.6850,0.8362,0.7042,0.9525,0.0056,476071
2,70_30,svm,12.456,0.6158,0.5046,0.8074,0.5468,0.9045,0.0013,476071
3,70_30,naive_bayes,5.415,0.4584,0.4519,0.6737,0.4665,0.8605,0.0007,476071
4,70_30,knn,5.188,0.7377,0.6758,0.8353,0.7011,0.9334,0.5380,476071
5,80_20,decision_tree,14.941,0.7355,0.6880,0.8044,0.7090,0.9211,0.0003,317381
6,80_20,random_forest,35.224,0.7582,0.6536,0.8310,0.6782,0.9527,0.0063,317381
7,80_20,svm,10.452,0.6158,0.4741,0.7950,0.5145,0.9045,0.0018,317381
8,80_20,naive_bayes,4.549,0.5480,0.4645,0.6548,0.4684,0.8665,0.0007,317381
9,80_20,knn,4.241,0.7398,0.6670,0.8354,0.6964,0.9336,0.5422,317381


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [5]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 21:10:45,615 INFO nb04: START 5-fold stratified CV, all models
2026-07-06 21:10:59,392 INFO ids.evaluation.cross_val: CV decision_tree fold 1/5: f1_macro=0.6828 acc=0.7219 (12.7s)
2026-07-06 21:11:12,061 INFO ids.evaluation.cross_val: CV decision_tree fold 2/5: f1_macro=0.7042 acc=0.7172 (11.9s)
2026-07-06 21:11:25,377 INFO ids.evaluation.cross_val: CV decision_tree fold 3/5: f1_macro=0.6793 acc=0.7293 (12.4s)
2026-07-06 21:11:38,519 INFO ids.evaluation.cross_val: CV decision_tree fold 4/5: f1_macro=0.6875 acc=0.7277 (12.4s)
2026-07-06 21:11:51,801 INFO ids.evaluation.cross_val: CV decision_tree fold 5/5: f1_macro=0.6961 acc=0.7318 (12.5s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   10.2s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   26.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.0s finished


2026-07-06 21:12:24,935 INFO ids.evaluation.cross_val: CV random_forest fold 1/5: f1_macro=0.6586 acc=0.7432 (30.1s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   26.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 21:12:57,768 INFO ids.evaluation.cross_val: CV random_forest fold 2/5: f1_macro=0.6664 acc=0.7451 (29.6s)


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   26.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.0s finished


2026-07-06 21:14:04,745 INFO ids.evaluation.cross_val: CV random_forest fold 4/5: f1_macro=0.6602 acc=0.7412 (30.2s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   27.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished


2026-07-06 21:14:39,102 INFO ids.evaluation.cross_val: CV random_forest fold 5/5: f1_macro=0.6647 acc=0.7420 (30.6s)
2026-07-06 21:14:52,216 INFO ids.evaluation.cross_val: CV svm fold 1/5: f1_macro=0.5367 acc=0.6127 (10.7s)
2026-07-06 21:15:04,584 INFO ids.evaluation.cross_val: CV svm fold 2/5: f1_macro=0.5341 acc=0.6272 (10.3s)
2026-07-06 21:15:16,641 INFO ids.evaluation.cross_val: CV svm fold 3/5: f1_macro=0.5369 acc=0.6126 (10.0s)
2026-07-06 21:15:27,295 INFO ids.evaluation.cross_val: CV svm fold 4/5: f1_macro=0.5399 acc=0.6138 (8.7s)
2026-07-06 21:15:38,229 INFO ids.evaluation.cross_val: CV svm fold 5/5: f1_macro=0.5445 acc=0.6219 (9.1s)
2026-07-06 21:15:43,264 INFO ids.evaluation.cross_val: CV naive_bayes fold 1/5: f1_macro=0.4507 acc=0.3796 (3.5s)
2026-07-06 21:15:48,282 INFO ids.evaluation.cross_val: CV naive_bayes fold 2/5: f1_macro=0.4611 acc=0.7568 (3.5s)
2026-07-06 21:15:53,326 INFO ids.evaluation.cross_val: CV naive_bayes fold 3/5: f1_macro=0.4197 acc=0.2337 (3.7s)
2026-07-

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.725575,0.005939,0.665315,0.011799,0.845481,0.028190,0.689990,0.010148,0.925630,0.004808
1,knn,0.739164,0.001067,0.672408,0.017053,0.841473,0.042015,0.693250,0.016298,0.933337,0.000701
2,naive_bayes,0.468324,0.196986,0.468019,0.011691,0.653238,0.034324,0.453769,0.020447,0.861136,0.002956
3,random_forest,0.742804,0.001468,0.643315,0.003611,0.858013,0.031637,0.661218,0.004266,0.955403,0.000344
4,svm,0.617616,0.006597,0.499592,0.000937,0.804934,0.024534,0.538416,0.003976,0.901292,0.001384


In [6]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 21:43:21,026 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40_rfe/cic2018/runs
